In [1]:
#In the following sections, we’ll build a neural network to classify images in the FashionMNIST dataset.
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## Get Device for Training

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


## Define the Class
### We define our neural network by subclassing nn.Module, and initialize the neural network layers in __init__. Every nn.Module subclass implements the operations on input data in the forward method.

In [8]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [10]:
#We create an instance of NeuralNetwork, and move it to the device, and print its structure.
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [12]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([4])


## Model Layers

In [15]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


## nn.Flatten

In [18]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


## nn.Linear

In [21]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


## nn.ReLU

In [24]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.2261,  0.0026, -0.6845, -0.1379,  0.2764, -0.1750,  0.6402, -0.1495,
          0.3059, -0.2254, -0.0655, -0.0585, -0.2402, -0.2387,  0.0990, -0.2691,
         -0.1604, -0.3767,  0.1584, -0.0475],
        [ 0.2152,  0.2659, -0.2346,  0.1621,  0.0024,  0.0311,  0.7723,  0.0986,
          0.3875, -0.0657,  0.2189,  0.0781, -0.3763, -0.1974, -0.1652, -0.4247,
         -0.0818, -0.4299,  0.1005,  0.2142],
        [ 0.0387,  0.2071, -0.4832,  0.2399, -0.1514,  0.0294,  0.9526, -0.1499,
          0.3809,  0.2224, -0.1129, -0.6129, -0.0886,  0.0267,  0.0277, -0.4460,
         -0.0662, -0.1245, -0.1833,  0.1318]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.2261, 0.0026, 0.0000, 0.0000, 0.2764, 0.0000, 0.6402, 0.0000, 0.3059,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0990, 0.0000, 0.0000, 0.0000,
         0.1584, 0.0000],
        [0.2152, 0.2659, 0.0000, 0.1621, 0.0024, 0.0311, 0.7723, 0.0986, 0.3875,
         0.0000, 0.2189, 0.0781, 0.0000, 0.0000, 0.00

## nn.Sequential

In [27]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

## nn.Softmax

In [30]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

## Model Parameters

In [33]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0011, -0.0251, -0.0236,  ...,  0.0337, -0.0299, -0.0072],
        [-0.0067,  0.0038,  0.0218,  ...,  0.0329,  0.0172,  0.0172]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0151, 0.0227], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0166, -0.0245,  0.0017,  ..., -0.0235, -0.0054, -0.0267],
        [ 0.0132, -0.0343,  0.0244,  ...,  0.0272,  0.0390,  0.0030]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si